 In this lab, you will build a small system that takes customer messages as input and generates summaries and suggested actions.
 You will explore zero-shot, one-shot, and few-shot prompting, and learn how prompt engineering and in-context learning affect outputs.

 Table of Contents
 - [1 - Set up Required Dependencies](#1)
 - [2 - Define Customer Messages Dataset](#2)
 - [3 - Zero-Shot Summarization](#3)
 - [4 - One-Shot Summarization](#4)
 - [5 - Few-Shot Summarization](#5)
- [6 - Adding Action Extraction](#6)
 - [7 - Evaluation: Comparing Prompt Types](#7)
- [8 - Generative Configuration Parameters for Inference](#8)
 - [9 - Prompt Experiments & In-Context Learning](#9)




1 - Set up Required Dependencies

In [1]:
# !pip install -U torch datasets transformers pandas

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd

# Load the model and tokenizer

In [4]:
model_name = "google/flan-t5-base"
model_name

'google/flan-t5-base'

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [6]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

#2 - Define Customer Messages Dataset

In [7]:
customer_messages = [
    "Hi, my internet has been very slow since yesterday and I can’t attend my meetings.",
    "I was charged twice for my subscription this month. Please fix this.",
    "The app crashes every time I try to upload a file."
]


In [8]:
df= pd.DataFrame(customer_messages, columns=["customer_message"])
df

,customer_message
0,"Hi, my internet has been very slow since yeste..."
1,I was charged twice for my subscription this m...
2,The app crashes every time I try to upload a f...


## 3 - Zero-Shot Summarization:
Explanation: Zero-shot relies entirely on the model's general knowledge. No examples are provided, only instructions.

In [9]:
zero_shot_summaries = []

In [10]:
for msg in customer_messages:
    prompt = f"Summarize the following customer message:\n\n{msg}"
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(**inputs)
    summary = tokenizer.decode(output[0], skip_special_tokens=True)
    zero_shot_summaries.append(summary)
df["zero_shot_summary"] = zero_shot_summaries
df

,customer_message,zero_shot_summary
0,"Hi, my internet has been very slow since yeste...",Is there anything I can do to help?
1,I was charged twice for my subscription this m...,I was charged twice for my subscription this m...
2,The app crashes every time I try to upload a f...,Is there a way to fix this?


In [12]:
prompt = f"Summarize the following customer message:\n\n{msg}"
inputs = tokenizer(prompt, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[12198,  1635,  1737,     8,   826,   884,  1569,    10,    37,  1120,
         22298,   334,    97,    27,   653,    12,  5940,     3,     9,  1042,
             5,     1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [95]:
output = model.generate(**inputs)
print(output)

tensor([[   0,   27,    7,  132,    3,    9,  194,   12, 2210,   48,   58,    1]])


In [96]:
summary = tokenizer.decode(output[0], skip_special_tokens=True)
summary

'Is there a way to fix this?'

In [97]:
zero_shot_summaries.append(summary)

In [98]:
zero_shot_summaries

['Is there anything I can do to help?',
 'I was charged twice for my subscription this month.',
 'Is there a way to fix this?',
 'Is there a way to fix this?']

#\4 - One-Shot Summarization
Explanation: One-shot provides a single reference example so the model can follow the style and structure.

In [11]:
one_shot_example = "Customer: My app crashes during upload.\nSummary: App crashes during file upload."


In [12]:
one_shot_summaries = []


In [13]:
prompt = f"""
Example:
{one_shot_example}

Now summarize this customer message:
Customer: {one_shot_example}
"""

In [15]:
inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(**inputs)
summary = tokenizer.decode(output[0], skip_special_tokens=True)
one_shot_summaries.append(summary)

In [103]:
one_shot_summaries

['The app crashes during upload.']

In [14]:
for msg in customer_messages:
    prompt = f"""
    Example:
    {one_shot_example}

    Now summarize this customer message:
    Customer: {msg}
    """
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(**inputs)
    summary = tokenizer.decode(output[0], skip_special_tokens=True)
    one_shot_summaries.append(summary)
df["one_shot_summaries"]=one_shot_summaries
df

,customer_message,zero_shot_summary,one_shot_summaries
0,"Hi, my internet has been very slow since yeste...",Is there anything I can do to help?,Can you help me?
1,I was charged twice for my subscription this m...,I was charged twice for my subscription this m...,Customer: I was charged twice for my subscript...
2,The app crashes every time I try to upload a f...,Is there a way to fix this?,The app crashes every time I try to upload a f...


#3-Few-shot
Explanation: Few-shot provides multiple examples, helping the model generalize style and structure better.

In [15]:
few_shot_examples = """
Example 1:
Customer: My internet is very slow.
Summary: Internet speed issue.

Example 2:
Customer: The app crashes when uploading files.
Summary: App crashes during upload.
"""


In [16]:
few_shot_summaries = []

In [17]:
for msg in customer_messages:
    prompt = f"""
{few_shot_examples}

Now summarize this customer message:
Customer: {msg}
"""
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(**inputs)
    summary = tokenizer.decode(output[0], skip_special_tokens=True)
    few_shot_summaries.append(summary)

df["few_shot_summary"] = few_shot_summaries
df

,customer_message,zero_shot_summary,one_shot_summaries,few_shot_summary
0,"Hi, my internet has been very slow since yeste...",Is there anything I can do to help?,Can you help me?,Summary: Internet speed issue
1,I was charged twice for my subscription this m...,I was charged twice for my subscription this m...,Customer: I was charged twice for my subscript...,Summary: Customer: I was charged twice for my ...
2,The app crashes every time I try to upload a f...,Is there a way to fix this?,The app crashes every time I try to upload a f...,Summary: App crashes every time I try to uploa...


#6 - Adding Action Extraction

In [18]:
few_shot_action_examples = """
Example 1:
Customer: My internet is very slow.
Summary: Internet speed issue.
Action: Ask user to restart router and check speed.

Example 2:
Customer: The app crashes when uploading files.
Summary: App crashes during upload.
Action: Ask user to update the app and retry upload.
"""

In [19]:
action_outputs = []

In [20]:
action_outputs = []

for msg in customer_messages:
    prompt = f"""
You are a support assistant.

{few_shot_action_examples}

Now process this customer message:
Customer: {msg}
"""
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(**inputs, max_new_tokens=50)
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    action_outputs.append(text)

df["few_shot_with_action"] = action_outputs
df

,customer_message,zero_shot_summary,one_shot_summaries,few_shot_summary,few_shot_with_action
0,"Hi, my internet has been very slow since yeste...",Is there anything I can do to help?,Can you help me?,Summary: Internet speed issue,Summary: Please help me with this.
1,I was charged twice for my subscription this m...,I was charged twice for my subscription this m...,Customer: I was charged twice for my subscript...,Summary: Customer: I was charged twice for my ...,Summary: Customer: I was charged twice for my ...
2,The app crashes every time I try to upload a f...,Is there a way to fix this?,The app crashes every time I try to upload a f...,Summary: App crashes every time I try to uploa...,Summary: Action: Ask user to update the app an...


# 7 - Evaluation: Comparing Prompt Types

In [21]:
df

,customer_message,zero_shot_summary,one_shot_summaries,few_shot_summary,few_shot_with_action
0,"Hi, my internet has been very slow since yeste...",Is there anything I can do to help?,Can you help me?,Summary: Internet speed issue,Summary: Please help me with this.
1,I was charged twice for my subscription this m...,I was charged twice for my subscription this m...,Customer: I was charged twice for my subscript...,Summary: Customer: I was charged twice for my ...,Summary: Customer: I was charged twice for my ...
2,The app crashes every time I try to upload a f...,Is there a way to fix this?,The app crashes every time I try to upload a f...,Summary: App crashes every time I try to uploa...,Summary: Action: Ask user to update the app an...


# 8 - Generative Configuration Parameters for Inference
Explanation: max_new_tokens → output length, temperature → randomness, top_p → token probability, affecting summary style and length.

In [22]:
prompt = "Summarize this customer message:\nCustomer: The app crashes every time I upload a file."

inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    temperature=0.7,
    top_p=0.9,
)

summary = tokenizer.decode(output[0], skip_special_tokens=True)
print(summary)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Can't upload files.


#9 - Prompt Experiments & In-Context Learning
Empty string → may generate extra text

Ending with Summary: → concise output

Rephrased beginning → tone/style changes

Adding guidance → limits length & focuses on key info

In [23]:
msg = "The app crashes every time I upload a file."

# Experiment 1: Ending with empty string

In [24]:
prompt_empty = f"Summarize this customer message:\nCustomer: {msg}\n"
inputs_empty = tokenizer(prompt_empty, return_tensors="pt")
output_empty = model.generate(**inputs_empty, max_new_tokens=50)
summary_empty = tokenizer.decode(output_empty[0], skip_special_tokens=True)
print("1️⃣ Empty string ending:\n", summary_empty, "\n")


1️⃣ Empty string ending:
 It crashes every time I upload a file. 



# Experiment 2: Ending with 'Summary:'

In [25]:
prompt_summary = f"Summarize this customer message:\nCustomer: {msg}\nSummary:"
inputs_summary = tokenizer(prompt_summary, return_tensors="pt")
output_summary = model.generate(**inputs_summary, max_new_tokens=50)
summary_summary = tokenizer.decode(output_summary[0], skip_special_tokens=True)
print("2️⃣ Ending with 'Summary:':\n", summary_summary, "\n")

2️⃣ Ending with 'Summary:':
 Can't upload files. 



# Experiment 3: Rephrased beginning

In [26]:
prompt_rephrase = f"Write a concise summary of the following issue reported by a customer:\nCustomer: {msg}\nSummary:"
inputs_rephrase = tokenizer(prompt_rephrase, return_tensors="pt")
output_rephrase = model.generate(**inputs_rephrase, max_new_tokens=50)
summary_rephrase = tokenizer.decode(output_rephrase[0], skip_special_tokens=True)
print("3️⃣ Rephrased beginning:\n", summary_rephrase, "\n")

3️⃣ Rephrased beginning:
 The app crashes every time I upload a file. 



 Experiment 4: Style guidance

In [27]:
prompt_style = f"Provide a short and clear summary in 1-2 sentences:\nCustomer: {msg}\nSummary:"
inputs_style = tokenizer(prompt_style, return_tensors="pt")
output_style = model.generate(**inputs_style, max_new_tokens=50)
summary_style = tokenizer.decode(output_style[0], skip_special_tokens=True)
print("4️⃣ Style guidance (short & clear):\n", summary_style, "\n")

4️⃣ Style guidance (short & clear):
 The app crashes every time I upload a file. 



In [28]:
# trying limit

In [29]:
def count_tokens(text):
    return len(tokenizer.tokenize(text))


def run_prompt(prompt, max_new_tokens=50):
    print("Token count:", count_tokens(prompt))

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,      # IMPORTANT
        max_length=512        # model limit
    )

    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [30]:
msg = "The app crashes every time I upload a file."


long_examples = """
Example 1:
Customer: Issue A
Summary: A

Example 2:
Customer: Issue B
Summary: B

Example 3:
Customer: Issue C
Summary: C

Example 4:
Customer: Issue D
Summary: D

Example 5:
Customer: Issue E
Summary: E

Example 6:
Customer: Issue F
Summary: F
"""

prompt_too_long = f"""
{long_examples}

Now summarize:
Customer: {msg}
Summary:
"""

print("Token count:", count_tokens(prompt_too_long))
print(run_prompt(prompt_too_long))


Token count: 79
Token count: 79
It crashes every time I upload a file
